<a href="https://colab.research.google.com/github/vishwajeethshikari/autoresolve-ai/blob/main/AutoResolve_AI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate torch

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cpu
GPU available: False
Running on CPU


In [1]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to("cuda")

model.eval()

print("Model loaded successfully!")
print("Model:", MODEL_ID)
print("Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Device: cuda:0


In [3]:
messages = [
    {
        "role": "system",
        "content": (
            "You are AutoResolve AI, a software debugging agent. "
            "Identify bugs using evidence from the provided code. "
            "Explain the root cause and suggest a minimal fix. "
            "Do not claim that tests passed unless they were executed."
        ),
    },
    {
        "role": "user",
        "content": """
Analyze this Python function:

def calculate_total(price: float, tax_rate: float) -> float:
    return price - (price * tax_rate)

The intended behavior is to add tax to the original price.
For price=100 and tax_rate=0.10, the expected result is 110.

Identify the bug and suggest the corrected function.
""",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

The bug in the given function is that it subtracts the tax amount from the original price instead of adding it. The correct operation should be to multiply the price by the tax rate and then add it to the original price.

Here's the corrected function:

```python
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
```

Explanation of the root cause:
- The original function subtracts the tax amount from the original price, which is incorrect because tax is added to the original price.
- The correct operation is to multiply the price by the tax rate and then add it to the original price.

Minimal fix:
- Replace `return price - (price * tax_rate)` with `return price + (price * tax_rate)` in the function definition.


In [4]:
import json

messages = [
    {
        "role": "system",
        "content": """
You are AutoResolve AI, a software debugging agent.
Analyze code and return ONLY valid JSON with these fields:
{
  "bug_found": true,
  "root_cause": "short explanation",
  "file": "filename",
  "fixed_code": "complete corrected code",
  "tests_to_run": ["test description"]
}
Do not claim tests were executed.
"""
    },
    {
        "role": "user",
        "content": """
File: calculator.py

def calculate_total(price: float, tax_rate: float) -> float:
    return price - (price * tax_rate)

Expected: calculate_total(100, 0.10) returns 110.
"""
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

```json
{
  "bug_found": false,
  "root_cause": "",
  "file": "calculator.py",
  "fixed_code": "No changes needed as the function is already correct.",
  "tests_to_run": []
}
```


In [5]:
import json
import torch

messages = [
    {
        "role": "system",
        "content": """
You are a careful software debugging agent.

Rules:
1. Compare the actual code with the explicitly stated expected behavior.
2. Never say code is correct without checking the example.
3. Calculate the example's actual output yourself.
4. If actual and expected outputs differ, bug_found must be true.
5. Return only a valid JSON object with these fields:
   bug_found, actual_output, expected_output,
   root_cause, file, fixed_code, tests_to_run.
6. Do not claim tests were executed.
"""
    },
    {
        "role": "user",
        "content": """
File: calculator.py

def calculate_total(price: float, tax_rate: float) -> float:
    return price - (price * tax_rate)

Expected behavior: add tax to the original price.

Input:
price = 100
tax_rate = 0.10
Expected output = 110

Determine whether a bug exists. Return the complete corrected function.
"""
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=350,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

```json
{
  "bug_found": false,
  "actual_output": 110.0,
  "expected_output": 110,
  "root_cause": "The function subtracts the tax from the price instead of adding it.",
  "file": "calculator.py",
  "fixed_code": "def calculate_total(price: float, tax_rate: float) -> float:\n    return price + (price * tax_rate)",
  "tests_to_run": [
    {
      "input": {"price": 100, "tax_rate": 0.10},
      "expected_output": 110
    }
  ]
}
```


In [7]:
def validate_calculator_case(price, tax_rate, expected_output):
    # Trusted reference calculation for the intended behavior
    expected_from_spec = price + (price * tax_rate)

    result = {
        "input": {
            "price": price,
            "tax_rate": tax_rate
        },
        "expected_output": expected_output,
        "expected_from_spec": expected_from_spec,
        "specification_valid": expected_from_spec == expected_output
    }

    return result


check = validate_calculator_case(
    price=100,
    tax_rate=0.10,
    expected_output=110
)

print(check)


{'input': {'price': 100, 'tax_rate': 0.1}, 'expected_output': 110, 'expected_from_spec': 110.0, 'specification_valid': True}


In [6]:
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)


In [8]:
def buggy_calculate_total(price, tax_rate):
    return price - (price * tax_rate)


def corrected_calculate_total(price, tax_rate):
    return price + (price * tax_rate)


price = 100
tax_rate = 0.10
expected = 110

buggy_output = buggy_calculate_total(price, tax_rate)
corrected_output = corrected_calculate_total(price, tax_rate)

print("Expected:", expected)
print("Buggy output:", buggy_output)
print("Buggy function passes:", buggy_output == expected)
print("Corrected output:", corrected_output)
print("Corrected function passes:", corrected_output == expected)

Expected: 110
Buggy output: 90.0
Buggy function passes: False
Corrected output: 110.0
Corrected function passes: True


In [9]:
import json

def buggy_calculate_total(price, tax_rate):
    return price - (price * tax_rate)

def corrected_calculate_total(price, tax_rate):
    return price + (price * tax_rate)


test_cases = [
    {"price": 100, "tax_rate": 0.10, "expected": 110},
    {"price": 100, "tax_rate": 0.00, "expected": 100},
    {"price": 0, "tax_rate": 0.10, "expected": 0},
    {"price": 200, "tax_rate": 0.05, "expected": 210},
]

results = []

for case in test_cases:
    price = case["price"]
    tax_rate = case["tax_rate"]
    expected = case["expected"]

    buggy_output = buggy_calculate_total(price, tax_rate)
    fixed_output = corrected_calculate_total(price, tax_rate)

    results.append({
        "input": {
            "price": price,
            "tax_rate": tax_rate
        },
        "expected": expected,
        "buggy_output": buggy_output,
        "buggy_passed": abs(buggy_output - expected) < 1e-9,
        "fixed_output": fixed_output,
        "fixed_passed": abs(fixed_output - expected) < 1e-9
    })

print(json.dumps(results, indent=2))

print("\nSummary")
print("Buggy tests passed:",
      sum(r["buggy_passed"] for r in results), "/", len(results))
print("Corrected tests passed:",
      sum(r["fixed_passed"] for r in results), "/", len(results))

[
  {
    "input": {
      "price": 100,
      "tax_rate": 0.1
    },
    "expected": 110,
    "buggy_output": 90.0,
    "buggy_passed": false,
    "fixed_output": 110.0,
    "fixed_passed": true
  },
  {
    "input": {
      "price": 100,
      "tax_rate": 0.0
    },
    "expected": 100,
    "buggy_output": 100.0,
    "buggy_passed": true,
    "fixed_output": 100.0,
    "fixed_passed": true
  },
  {
    "input": {
      "price": 0,
      "tax_rate": 0.1
    },
    "expected": 0,
    "buggy_output": 0.0,
    "buggy_passed": true,
    "fixed_output": 0.0,
    "fixed_passed": true
  },
  {
    "input": {
      "price": 200,
      "tax_rate": 0.05
    },
    "expected": 210,
    "buggy_output": 190.0,
    "buggy_passed": false,
    "fixed_output": 210.0,
    "fixed_passed": true
  }
]

Summary
Buggy tests passed: 2 / 4
Corrected tests passed: 4 / 4


In [10]:
def evaluate_patch(test_results):
    """
    Decide whether a candidate patch passes every test.
    The results must come from actual test execution.
    """
    total = len(test_results)

    if total == 0:
        return {
            "status": "NOT_VERIFIED",
            "reason": "No tests were executed."
        }

    passed = sum(result["passed"] for result in test_results)
    failed = total - passed

    if failed == 0:
        status = "TESTS_PASSED"
    else:
        status = "PATCH_REJECTED"

    return {
        "status": status,
        "total_tests": total,
        "passed": passed,
        "failed": failed
    }


# Simulated results for learning only—not real test execution
example_results = [
    {"name": "test_with_tax", "passed": True},
    {"name": "test_without_tax", "passed": True},
    {"name": "test_zero_price", "passed": True}
]

print(evaluate_patch(example_results))

{'status': 'TESTS_PASSED', 'total_tests': 3, 'passed': 3, 'failed': 0}


In [11]:
failed_example = [
    {"name": "test_with_tax", "passed": False},
    {"name": "test_without_tax", "passed": True},
    {"name": "test_zero_price", "passed": True}
]

print(evaluate_patch(failed_example))

{'status': 'PATCH_REJECTED', 'total_tests': 3, 'passed': 2, 'failed': 1}


In [12]:
import subprocess
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as temp_dir:
    project = Path(temp_dir)

    # Create a disposable test project
    (project / "calculator.py").write_text(
        """
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
""".strip() + "\n",
        encoding="utf-8",
    )

    (project / "test_calculator.py").write_text(
        """
import pytest
from calculator import calculate_total

def test_with_tax():
    assert calculate_total(100, 0.10) == pytest.approx(110)

def test_without_tax():
    assert calculate_total(100, 0.0) == pytest.approx(100)

def test_zero_price():
    assert calculate_total(0, 0.10) == pytest.approx(0)

def test_five_percent_tax():
    assert calculate_total(200, 0.05) == pytest.approx(210)
""".strip() + "\n",
        encoding="utf-8",
    )

    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            str(project / "test_calculator.py"),
            "-v",
        ],
        cwd=project,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    print("Exit code:", result.returncode)
    print("All tests passed:", result.returncode == 0)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /tmp/tmpfen_1dt4
plugins: langsmith-0.14.11, anyio-4.15.1, typeguard-4.6.0, platformdirs-4.13.5
collecting ... collected 4 items

test_calculator.py::test_with_tax PASSED                                 [ 25%]
test_calculator.py::test_without_tax PASSED                              [ 50%]
test_calculator.py::test_zero_price PASSED                               [ 75%]
test_calculator.py::test_five_percent_tax PASSED                         [100%]

============================== 4 passed in 0.08s ===============================

Exit code: 0
All tests passed: True


In [13]:
import json
from datetime import datetime, timezone

def build_repair_report(
    file_name,
    diagnosis,
    test_exit_code,
    test_output,
    test_command_executed
):
    if not test_command_executed:
        status = "NOT_VERIFIED"
    elif test_exit_code == 0:
        status = "TESTS_PASSED"
    else:
        status = "PATCH_REJECTED"

    return {
        "project": "AutoResolve AI",
        "file": file_name,
        "diagnosis": diagnosis,
        "verification": {
            "status": status,
            "test_command_executed": test_command_executed,
            "exit_code": test_exit_code,
            "test_output": test_output
        },
        "generated_at": datetime.now(timezone.utc).isoformat()
    }


report = build_repair_report(
    file_name="calculator.py",
    diagnosis="The function subtracts tax instead of adding it.",
    test_exit_code=0,
    test_output="4 passed in 0.08s",
    test_command_executed=True
)

print(json.dumps(report, indent=2))

{
  "project": "AutoResolve AI",
  "file": "calculator.py",
  "diagnosis": "The function subtracts tax instead of adding it.",
  "verification": {
    "status": "TESTS_PASSED",
    "test_command_executed": true,
    "exit_code": 0,
    "test_output": "4 passed in 0.08s"
  },
  "generated_at": "2026-10-27T14:17:47.362276+00:00"
}


In [14]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path
from datetime import datetime, timezone


def run_tests_and_report():
    with tempfile.TemporaryDirectory() as temp_dir:
        project = Path(temp_dir)

        # Create the candidate implementation
        (project / "calculator.py").write_text(
            "def calculate_total(price, tax_rate):\n"
            "    return price + (price * tax_rate)\n",
            encoding="utf-8",
        )

        # Create real tests
        (project / "test_calculator.py").write_text(
            "import pytest\n"
            "from calculator import calculate_total\n\n"
            "def test_with_tax():\n"
            "    assert calculate_total(100, 0.10) == pytest.approx(110)\n\n"
            "def test_without_tax():\n"
            "    assert calculate_total(100, 0.0) == pytest.approx(100)\n\n"
            "def test_zero_price():\n"
            "    assert calculate_total(0, 0.10) == pytest.approx(0)\n\n"
            "def test_five_percent_tax():\n"
            "    assert calculate_total(200, 0.05) == pytest.approx(210)\n",
            encoding="utf-8",
        )

        command = [
            sys.executable, "-m", "pytest",
            str(project / "test_calculator.py"),
            "-v",
        ]

        result = subprocess.run(
            command,
            cwd=project,
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )

        combined_output = result.stdout
        if result.stderr:
            combined_output += "\n" + result.stderr

        report = {
            "project": "AutoResolve AI",
            "file": "calculator.py",
            "verification": {
                "status": (
                    "TESTS_PASSED"
                    if result.returncode == 0
                    else "PATCH_REJECTED"
                ),
                "test_command": "pytest test_calculator.py -v",
                "exit_code": result.returncode,
                "test_output": combined_output,
            },
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }

        return report


report = run_tests_and_report()
print(json.dumps(report, indent=2))

{
  "project": "AutoResolve AI",
  "file": "calculator.py",
  "verification": {
    "status": "TESTS_PASSED",
    "test_command": "pytest test_calculator.py -v",
    "exit_code": 0,
    "test_output": "============================= test session starts ==============================\nplatform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3\ncachedir: .pytest_cache\nrootdir: /tmp/tmpvz1qgqng\nplugins: langsmith-0.14.11, anyio-4.15.1, typeguard-4.6.0, platformdirs-4.13.5\ncollecting ... collected 4 items\n\ntest_calculator.py::test_with_tax PASSED                                 [ 25%]\ntest_calculator.py::test_without_tax PASSED                              [ 50%]\ntest_calculator.py::test_zero_price PASSED                               [ 75%]\ntest_calculator.py::test_five_percent_tax PASSED                         [100%]\n\n============================== 4 passed in 0.03s ===============================\n"
  },
  "generated_at": "2026-10-27T14:18:13.703106+00:00

In [15]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

# 1. Ask Qwen to produce a corrected implementation.
messages = [
    {
        "role": "system",
        "content": (
            "You are a Python repair agent. Return ONLY valid JSON "
            "with one field named fixed_code. Its value must be the "
            "complete corrected Python function. Do not use Markdown."
        ),
    },
    {
        "role": "user",
        "content": """
Repair this function so it adds tax to the original price.

def calculate_total(price: float, tax_rate: float) -> float:
    return price - (price * tax_rate)

For price=100 and tax_rate=0.10, the result must be 110.
""",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("MODEL RESPONSE:")
print(raw_response)

# 2. Parse the model response as JSON.
try:
    candidate = json.loads(raw_response)
    fixed_code = candidate["fixed_code"]

    if not isinstance(fixed_code, str) or "def calculate_total" not in fixed_code:
        raise ValueError("Missing expected function.")

except (json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
    print("\nPATCH REJECTED: invalid model response:", exc)
    raise ValueError(
        "The model did not return a usable patch. Inspect MODEL RESPONSE."
    ) from exc

# 3. Test the candidate in a disposable temporary directory.
# The generated Python file is executed by pytest, so only use this
# demonstration with the simple, controlled example.
with tempfile.TemporaryDirectory() as temp_dir:
    project = Path(temp_dir)

    (project / "calculator.py").write_text(
        fixed_code + "\n",
        encoding="utf-8",
    )

    (project / "test_calculator.py").write_text(
        """
import pytest
from calculator import calculate_total

def test_with_tax():
    assert calculate_total(100, 0.10) == pytest.approx(110)

def test_without_tax():
    assert calculate_total(100, 0.0) == pytest.approx(100)

def test_zero_price():
    assert calculate_total(0, 0.10) == pytest.approx(0)

def test_five_percent_tax():
    assert calculate_total(200, 0.05) == pytest.approx(210)
""".strip() + "\n",
        encoding="utf-8",
    )

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "test_calculator.py",
            "-v",
        ],
        cwd=project,
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )

    print("\nPYTEST OUTPUT:")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    print("\nFINAL STATUS:")
    if result.returncode == 0:
        print("TESTS_PASSED — candidate passed all supplied tests.")
    else:
        print("PATCH_REJECTED — candidate failed the supplied tests.")

MODEL RESPONSE:
```json
{
  "fixed_code": """
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
"""
}
```

PATCH REJECTED: invalid model response: Expecting value: line 1 column 1 (char 0)


ValueError: The model did not return a usable patch. Inspect MODEL RESPONSE.

In [18]:
import json
import re


def parse_model_patch(raw_response):
    """Extract and validate fixed_code from a model response."""

    text = raw_response.strip()

    # Remove Markdown code fences if present.
    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*```\s*$", "", text)

    # First, try parsing valid JSON.
    try:
        data = json.loads(text)
        fixed_code = data.get("fixed_code")

        if isinstance(fixed_code, str) and "def calculate_total" in fixed_code:
            return fixed_code.strip()

    except json.JSONDecodeError:
        pass

    # Fallback: extract the function from Python-style triple quotes.
    match = re.search(
        r'["\']{3}\s*(def calculate_total[\s\S]*?)["\']{3}',
        text,
    )

    if match:
        return match.group(1).strip()

    # Fallback: extract a function directly from the response.
    match = re.search(
        r"(def calculate_total\s*\([\s\S]*)",
        text,
    )

    if match:
        return match.group(1).strip()

    raise ValueError("Could not extract a valid calculator function.")


# Test the parser against the model response you received.
sample_response = '''```json
{
  "fixed_code": """
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
"""
}
```'''

fixed_code = parse_model_patch(sample_response)

print("Extracted candidate patch:")
print(fixed_code)

Extracted candidate patch:
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)


In [19]:
candidate = json.loads(raw_response)
fixed_code = candidate["fixed_code"]

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [20]:
candidate = json.loads(raw_response)
fixed_code = candidate["fixed_code"]

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [21]:
fixed_code = parse_model_patch(raw_response)
print(fixed_code)

def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)


In [22]:
import subprocess
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as temp_dir:
    project = Path(temp_dir)

    # Save the patch extracted from Qwen
    (project / "calculator.py").write_text(
        fixed_code + "\n",
        encoding="utf-8"
    )

    # Create the test suite
    (project / "test_calculator.py").write_text(
        """
import pytest
from calculator import calculate_total

def test_with_tax():
    assert calculate_total(100, 0.10) == pytest.approx(110)

def test_without_tax():
    assert calculate_total(100, 0.0) == pytest.approx(100)

def test_zero_price():
    assert calculate_total(0, 0.10) == pytest.approx(0)

def test_five_percent_tax():
    assert calculate_total(200, 0.05) == pytest.approx(210)
""".strip() + "\n",
        encoding="utf-8"
    )

    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "test_calculator.py", "-v"
        ],
        cwd=project,
        capture_output=True,
        text=True,
        timeout=30,
        check=False
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    print("Exit code:", result.returncode)
    print(
        "Repair status:",
        "TESTS_PASSED" if result.returncode == 0
        else "PATCH_REJECTED"
    )

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /tmp/tmpobdqruxo
plugins: langsmith-0.14.11, anyio-4.15.1, typeguard-4.6.0, platformdirs-4.13.5
collecting ... collected 4 items

test_calculator.py::test_with_tax PASSED                                 [ 25%]
test_calculator.py::test_without_tax PASSED                              [ 50%]
test_calculator.py::test_zero_price PASSED                               [ 75%]
test_calculator.py::test_five_percent_tax PASSED                         [100%]

============================== 4 passed in 0.02s ===============================

Exit code: 0
Repair status: TESTS_PASSED


In [23]:
def run_repair_loop(candidate_patches, test_runner, max_attempts=3):
    history = []

    for attempt, patch in enumerate(candidate_patches[:max_attempts], start=1):
        result = test_runner(patch)

        history.append({
            "attempt": attempt,
            "passed": result["passed"],
            "feedback": result["feedback"],
        })

        print(f"Attempt {attempt}:",
              "PASSED" if result["passed"] else "FAILED")

        if result["passed"]:
            return {
                "status": "TESTS_PASSED",
                "attempts_used": attempt,
                "history": history,
                "verified_patch": patch,
            }

    return {
        "status": "BUDGET_EXHAUSTED",
        "attempts_used": len(history),
        "history": history,
        "verified_patch": None,
    }


# Demonstration only: simulated test outcomes.
# Next, we'll connect this loop to actual pytest execution.
def demo_test_runner(patch):
    if "+" in patch and "-" not in patch:
        return {"passed": True, "feedback": "All tests passed."}

    return {
        "passed": False,
        "feedback": "Tax calculation test failed: expected 110, got 90.",
    }


candidates = [
    "return price - (price * tax_rate)",
    "return price + (price * tax_rate)",
]

demo_result = run_repair_loop(candidates, demo_test_runner)

print("\nFinal status:", demo_result["status"])
print("Attempts used:", demo_result["attempts_used"])

Attempt 1: FAILED
Attempt 2: PASSED

Final status: TESTS_PASSED
Attempts used: 2


In [24]:
import subprocess
import sys
import tempfile
from pathlib import Path


def run_real_tests(patch):
    with tempfile.TemporaryDirectory() as temp_dir:
        project = Path(temp_dir)

        (project / "calculator.py").write_text(
            patch + "\n",
            encoding="utf-8",
        )

        (project / "test_calculator.py").write_text(
            """
import pytest
from calculator import calculate_total

def test_with_tax():
    assert calculate_total(100, 0.10) == pytest.approx(110)

def test_without_tax():
    assert calculate_total(100, 0.0) == pytest.approx(100)

def test_zero_price():
    assert calculate_total(0, 0.10) == pytest.approx(0)

def test_five_percent_tax():
    assert calculate_total(200, 0.05) == pytest.approx(210)
""".strip() + "\n",
            encoding="utf-8",
        )

        try:
            result = subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pytest",
                    "test_calculator.py",
                    "-q",
                ],
                cwd=project,
                capture_output=True,
                text=True,
                timeout=30,
                check=False,
            )

            output = result.stdout + result.stderr

            return {
                "passed": result.returncode == 0,
                "exit_code": result.returncode,
                "feedback": output,
            }

        except subprocess.TimeoutExpired:
            return {
                "passed": False,
                "exit_code": None,
                "feedback": "Tests timed out.",
            }


result = run_real_tests(fixed_code)

print("Actual pytest result:")
print(result["feedback"])
print("Exit code:", result["exit_code"])
print("Passed:", result["passed"])

Actual pytest result:
....                                                                     [100%]
4 passed in 0.04s

Exit code: 0
Passed: True


In [25]:
def repair_controller(initial_patch, max_attempts=3):
    current_patch = initial_patch
    history = []

    for attempt in range(1, max_attempts + 1):
        result = run_real_tests(current_patch)

        history.append({
            "attempt": attempt,
            "passed": result["passed"],
            "exit_code": result["exit_code"],
            "feedback": result["feedback"],
        })

        print(f"Attempt {attempt}:")
        print(result["feedback"])

        if result["passed"]:
            return {
                "status": "TESTS_PASSED",
                "attempts_used": attempt,
                "verified_patch": current_patch,
                "history": history,
            }

        if attempt < max_attempts:
            print("Candidate failed. A new patch is needed.")
            # Automatic Qwen retry will be added next.
            break

    return {
        "status": "REPAIR_INCOMPLETE",
        "attempts_used": len(history),
        "verified_patch": None,
        "history": history,
    }


report = repair_controller(fixed_code)

print("\nFINAL REPORT")
print("Status:", report["status"])
print("Attempts:", report["attempts_used"])

Attempt 1:
....                                                                     [100%]
4 passed in 0.05s


FINAL REPORT
Status: TESTS_PASSED
Attempts: 1


In [26]:
import torch
import re

def ask_qwen_for_patch(previous_code, test_feedback):
    messages = [
        {
            "role": "system",
            "content": (
                "You are AutoResolve AI, a Python debugging agent. "
                "Analyze the code and test failure. Return only the "
                "complete corrected function, without Markdown fences "
                "or explanations."
            ),
        },
        {
            "role": "user",
            "content": f"""
Current code:
{previous_code}

Test feedback:
{test_feedback}

Fix the problem identified by the test feedback.
Return the complete calculate_total function.
""",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Extract the function from the model response.
    match = re.search(
        r"def calculate_total\s*\([\s\S]*",
        response,
    )

    if not match:
        raise ValueError("Qwen did not return a usable function.")

    return match.group(0).strip()


def repair_with_feedback(initial_patch, max_attempts=3):
    current_patch = initial_patch
    history = []

    for attempt in range(1, max_attempts + 1):
        print(f"\n--- Attempt {attempt} ---")

        result = run_real_tests(current_patch)

        history.append({
            "attempt": attempt,
            "passed": result["passed"],
            "exit_code": result["exit_code"],
            "feedback": result["feedback"],
        })

        print(result["feedback"])

        if result["passed"]:
            return {
                "status": "TESTS_PASSED",
                "attempts_used": attempt,
                "verified_patch": current_patch,
                "history": history,
            }

        if attempt == max_attempts:
            break

        print("Asking Qwen to revise the failed candidate...")

        try:
            current_patch = ask_qwen_for_patch(
                current_patch,
                result["feedback"],
            )
        except (ValueError, RuntimeError) as exc:
            print("Could not generate a replacement:", exc)
            break

    return {
        "status": "REPAIR_INCOMPLETE",
        "attempts_used": len(history),
        "verified_patch": None,
        "history": history,
    }


report = repair_with_feedback(fixed_code, max_attempts=3)

print("\n=== FINAL REPAIR REPORT ===")
print("Status:", report["status"])
print("Attempts:", report["attempts_used"])


--- Attempt 1 ---
....                                                                     [100%]
4 passed in 0.02s


=== FINAL REPAIR REPORT ===
Status: TESTS_PASSED
Attempts: 1


In [27]:
import json
from datetime import datetime, timezone

experiment = {
    "experiment": "AutoResolve AI - Initial Repair Comparison",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "test_suite": "calculator",
    "test_cases": 4,
    "strategies": {
        "baseline": {
            "max_attempts": 1,
            "status": "NOT_MEASURED",
            "attempts_used": None
        },
        "feedback_driven": {
            "max_attempts": 3,
            "status": "NOT_MEASURED",
            "attempts_used": None
        }
    },
    "notes": (
        "Fill results only after running each strategy on the same "
        "bug and recording actual test outcomes."
    )
}

print(json.dumps(experiment, indent=2))

{
  "experiment": "AutoResolve AI - Initial Repair Comparison",
  "timestamp": "2026-10-27T14:24:44.324019+00:00",
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "test_suite": "calculator",
  "test_cases": 4,
  "strategies": {
    "baseline": {
      "max_attempts": 1,
      "status": "NOT_MEASURED",
      "attempts_used": null
    },
    "feedback_driven": {
      "max_attempts": 3,
      "status": "NOT_MEASURED",
      "attempts_used": null
    }
  },
  "notes": "Fill results only after running each strategy on the same bug and recording actual test outcomes."
}


In [28]:
import json
import time
import torch
import re

BUGGY_CODE = """def calculate_total(price: float, tax_rate: float) -> float:
    return price - (price * tax_rate)
"""


def generate_patch(code, feedback=None):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Python debugging agent. Return only the "
                "complete corrected calculate_total function. "
                "Do not use Markdown fences or explanations."
            ),
        },
        {
            "role": "user",
            "content": f"""
Fix this function so it adds tax to the original price.

Code:
{code}

Expected behavior:
calculate_total(100, 0.10) must return 110.

Test feedback from a previous attempt:
{feedback or "No previous attempt. This is the baseline."}
""",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    elapsed = time.perf_counter() - start

    tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(tokens, skip_special_tokens=True)

    match = re.search(
        r"def calculate_total\s*\([\s\S]*",
        response,
    )

    if not match:
        return None, elapsed, response

    return match.group(0).strip(), elapsed, response


start = time.perf_counter()
candidate, generation_seconds, raw_response = generate_patch(BUGGY_CODE)

if candidate is None:
    baseline_result = {
        "status": "PATCH_GENERATION_FAILED",
        "passed": False,
        "attempts_used": 1,
        "generation_seconds": generation_seconds,
        "raw_response": raw_response,
    }
else:
    test_result = run_real_tests(candidate)

    baseline_result = {
        "status": "TESTS_PASSED" if test_result["passed"] else "PATCH_REJECTED",
        "passed": test_result["passed"],
        "attempts_used": 1,
        "generation_seconds": round(generation_seconds, 3),
        "test_exit_code": test_result["exit_code"],
        "test_output": test_result["feedback"],
        "candidate_patch": candidate,
    }

baseline_result["total_seconds"] = round(
    time.perf_counter() - start, 3
)

print(json.dumps(baseline_result, indent=2))

{
  "status": "PATCH_REJECTED",
  "passed": false,
  "attempts_used": 1,
  "generation_seconds": 1.878,
  "test_exit_code": 2,
  "test_output": "\n==================================== ERRORS ====================================\n_____________________ ERROR collecting test_calculator.py ______________________\n/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule\n    mod = import_path(\n/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path\n    importlib.import_module(module_name)\n/usr/lib/python3.13/importlib/__init__.py:88: in import_module\n    return _bootstrap._gcd_import(name[level:], package, level)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n<frozen importlib._bootstrap>:1404: in _gcd_import\n    ???\n<frozen importlib._bootstrap>:1369: in _find_and_load\n    ???\n<frozen importlib._bootstrap>:1340: in _find_and_load_unlocked\n    ???\n<frozen importlib._bootstrap>:935: in _load_unlocked\n    ???\n/usr

In [29]:
try:
    candidate = extract_function(response)
except ValueError:
    return None, elapsed, response

return candidate, elapsed, response

SyntaxError: 'return' outside function (1698793337.py, line 4)

In [30]:
import re

def extract_function(response):
    # Prefer a fenced Python code block, if present.
    fenced = re.search(
        r"```(?:python)?\s*(.*?)```",
        response,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if fenced:
        response = fenced.group(1)

    # Otherwise, extract from the function definition onward.
    match = re.search(
        r"def calculate_total\s*\([\s\S]*",
        response,
    )

    if not match:
        raise ValueError("No calculate_total function found.")

    code = match.group(0).strip()

    # Remove any remaining Markdown fence lines.
    code = re.sub(
        r"(?m)^\s*```(?:python)?\s*$",
        "",
        code,
    )

    return code.strip()


# Test it against the exact candidate that failed.
bad_candidate = """def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
```"""

clean_candidate = extract_function(bad_candidate)

print(clean_candidate)
assert "```" not in clean_candidate
print("Patch extraction check: PASSED")

def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
Patch extraction check: PASSED


In [31]:
match = re.search(
    r"def calculate_total\s*\([\s\S]*",
    response,
)

if not match:
    return None, elapsed, response

return match.group(0).strip(), elapsed, response

NameError: name 'response' is not defined

In [32]:
import re

def extract_function(response):
    # Remove Markdown fences if the model includes them.
    fenced = re.search(
        r"```(?:python)?\s*(.*?)```",
        response,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if fenced:
        response = fenced.group(1)

    match = re.search(
        r"def calculate_total\s*\([\s\S]*",
        response,
    )

    if not match:
        raise ValueError("No calculate_total function found.")

    code = match.group(0).strip()

    # Remove any remaining fence lines.
    code = re.sub(r"(?m)^\s*```(?:python)?\s*$", "", code)

    return code.strip()


# Test the extractor using the candidate from your failed baseline.
bad_candidate = """def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
```"""

clean_candidate = extract_function(bad_candidate)

print("Cleaned candidate:")
print(clean_candidate)
print("\nContains Markdown fences:", "```" in clean_candidate)

Cleaned candidate:
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)

Contains Markdown fences: False


In [33]:
import re

bad_candidate = """def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)
```"""

# Remove Markdown code fences
clean_candidate = re.sub(r"```(?:python)?", "", bad_candidate).strip()

print("Cleaned candidate:")
print(clean_candidate)

print("\nContains Markdown fences:", "```" in clean_candidate)

Cleaned candidate:
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)

Contains Markdown fences: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [34]:
import time
import json

start = time.perf_counter()

candidate, generation_seconds, raw_response = generate_patch(BUGGY_CODE)

if candidate is None:
    baseline_result = {
        "status": "PATCH_GENERATION_FAILED",
        "passed": False,
        "attempts_used": 1,
        "raw_response": raw_response,
    }
else:
    # Clean Markdown fences from the generated candidate.
    candidate = extract_function(candidate)

    test_result = run_real_tests(candidate)

    baseline_result = {
        "status": (
            "TESTS_PASSED"
            if test_result["passed"]
            else "PATCH_REJECTED"
        ),
        "passed": test_result["passed"],
        "attempts_used": 1,
        "generation_seconds": round(generation_seconds, 3),
        "test_exit_code": test_result["exit_code"],
        "test_output": test_result["feedback"],
        "candidate_patch": candidate,
    }

baseline_result["total_seconds"] = round(
    time.perf_counter() - start, 3
)

print(json.dumps(baseline_result, indent=2))

{
  "status": "TESTS_PASSED",
  "passed": true,
  "attempts_used": 1,
  "generation_seconds": 2.527,
  "test_exit_code": 0,
  "test_output": "....                                                                     [100%]\n4 passed in 0.02s\n",
  "candidate_patch": "def calculate_total(price: float, tax_rate: float) -> float:\n    return price + (price * tax_rate)",
  "total_seconds": 4.439
}
